In [3]:
#Packages to Import

#Numerical Elements
from numpy.linalg import norm
import numpy as np
from numpy import dot, array, transpose, diag

#Fun Progress Bar
from tqdm.notebook import tqdm

#Misc System (plotting etc)
import sys
import matplotlib.image as mpimg
import matplotlib.pyplot as plt

import pickle
import warnings
warnings.filterwarnings('ignore')


#MCMC Sampliers and Related Utilities
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
%run ../MCMC_Sampliers.ipynb



#Plotting Libraries
import matplotlib.pyplot as plt
from numpy.linalg import norm



plt.rcParams.update({
    "text.usetex": True,
    "font.family": "sans-serif",
    "font.sans-serif": ["Helvetica"]})

In [34]:
from math import exp
import cupy as cp

def MpCN_GPU(q0,dim,Cov,rho,Pot,NProps,L):
    """
    The Multiproposal PcN Algorithm 
    Written for GPU Use
        Provdies to samples from measure of the form
        mu(dq) = exp( - Pot(q))mu_0(dq) where mu_0 = N(0, Cov)
        The imputs are
            q0 -initial value
            dim-dimension of the target meausre
            Cov-covariance of mu_0
            rho - algorithmic parameter taking values in [0,1)
            Pot- potential `loglikelihood' term in mu
            NProps-number of proposals per step ('p')
            L-total number of iteration steps
    """

    q0  = cp.asarray(q0,  dtype=cp.float64)
    Cov = cp.asarray(Cov, dtype=cp.float64)

    
    rng = cp.random.default_rng()
    samp = cp.empty((L + 1, dim), dtype=float) 
    #Make an array for the samples
    samp[0] = cp.array(q0)
    Cov_chol = cp.linalg.cholesky(Cov) 
    #Find E such that EE^* = C
    eta = np.sqrt(1.0 - rho * rho)
    for samID in range(1, L + 1):
        qtjCen = rho * samp[samID -1] + eta * Cov_chol @ rng.standard_normal(dim) 
        #draw initial center point
        curProps = cp.concatenate((samp[samID -1][:,None],rho* qtjCen[..., None] + eta * Cov_chol @ rng.standard_normal((dim,NProps))),axis =-1).T #cloud of proposals
        logAcp = cp.empty(NProps + 1, dtype=float)
        for j in range(NProps+1):
            logAcp[j] = -1*Pot(curProps[j])
        logAcp_max = cp.max(logAcp)
        Acp = cp.exp(logAcp - logAcp_max)  # stabilised weights
        probs = Acp/Acp.sum()

        u   = rng.random()                  # scalar in (0,1)
        idx = int(cp.searchsorted(cp.cumsum(probs), u))
        #idx = rng.choice(NProps+1, p=Acp / Acp.sum())
        samp[samID] = curProps[idx].copy()
    return cp.asnumpy(samp) 

#Numerical Set up for AD Toy Model
#AD Toy Model
#GPU Version

def MkAD_A_Mat(mod_dim: int, cur_apar: cp.ndarray) -> cp.ndarray:
    """
    Generate an antisymmetric matrix A given its strictly-upper-triangle
    parameters `cur_apar` (length = mod_dim*(mod_dim-1)//2).

    Returns: A (mod_dim × mod_dim) as cp.ndarray on the GPU.
    """
    A = cp.zeros((mod_dim, mod_dim), dtype=cp.float64)
    A[cp.triu_indices(mod_dim, k=1)] = cur_apar
    return A - A.T                 # ensure antisymmetry

# ------------------------------------------------------------------
# 2. Solve (A + κ I) θ = g
def getThA(mod_dim: int,
              apar: cp.ndarray,
              g: cp.ndarray,
              kappa: float) -> cp.ndarray:
    """
    Solve (A + kappa*I) θ = g on the GPU.
    """
    A_plus_kI = mk_AD_A_mat(mod_dim, apar) + kappa * cp.eye(mod_dim, dtype=cp.float64)
    return cp.linalg.solve(A_plus_kI, g)

# ------------------------------------------------------------------
# 3. Diagonal covariance matrix
def mkDiagCov(vars_: cp.ndarray) -> cp.ndarray:
    """Return diag(vars_) on the GPU."""
    return cp.diag(vars_.astype(cp.float64))

# ------------------------------------------------------------------
# 4. Uniform random orthogonal matrix via QR
def random_orth_matrix(n: int, dtype=cp.float64, rng=None) -> cp.ndarray:
    """
    Draw a matrix uniformly from O(n) using QR of a Gaussian matrix,
    entirely on the GPU.
    """
    rng = rng or cp.random.default_rng()
    A   = rng.standard_normal((n, n), dtype=dtype)
    Q, R = cp.linalg.qr(A)
    # Make Q uniformly distributed over O(n) (correct the sign ambiguity)
    d = cp.sign(cp.diag(R))
    Q *= d
    return Q


In [35]:
#Example 1: From the Parallel MCMC paper

#Specifying Problem Parameters


#Model Dimension and Parameter Size

ModDm = 4
NumParms = int(ModDm*(ModDm -1)/2)


#The Forward Model entails solving for th(A) for any antisymmetric A where
# (A + kap I) th = g 
# so that th(A) = th_{k,g}

# g = (g0,g1, g2, g3)^T

g0 = .1
g1 = 0
g2 = 5
g3 = 2
#g = np.transpose(np.array([g0,g1,g2,g3]))
g = cp.asarray([g0,g1,g2,g3],dtype=cp.float64)



# Coefficent of the `regularization/diffusion term'
kap = .05

#Specification `observed data' y0, y1
y0 = 4.601
y1 = 18.021

y = [y0,y1]

yData = cp.asarray([y0,y1],dtype=cp.float64)

# `observation noise coefficent'

sig = 2

# Covariance of the 'prior' C = cov0[1^{-gam}, 2^[-gam],..., N^{-gam}]
# where N is the number of parameters in the model NumParms = 6 
#for number of enetries that need to be specified in A

cov0 = 5
gam = 1.5

# Specify Potential and Prior Covariance
# The Posterior is of the form
# mu(dA) = Z^{-1} \exp( -1/(2 sig^2) ( (y0 - th(A)(0))^2 +(y1 - th(A)(0))^2 ) mu_0(dA)
# where
# mu_0(dA) = Z^{-1}_0 \exp( - 1/2<C^{-1}A, A>)

inv_two_sigma2 = 0.5 / (sig**2) 


Poty = lambda a : inv_two_sigma2*cp.linalg.norm(yData - getThA(ModDm, a, g, kap)[:2])**2
CovDiag = cp.asarray(
    [cov0 * (j ** (-gam)) for j in range(1, NumParms + 1)],
    dtype=cp.float64,          # pick float32 if that’s what you use elsewhere
)
Cov = mkDiagCov(CovDiag)

In [ ]:
#Generate Data Using MpCN

#Specifying File Location to Save Data 

FileNmBase= "Data/Advection_Diffusion_Toy/"

paraStr = "gvec_" + str(g) + "_kap_" + str(kap) + "_y_" + str(y)+ "_sig_" + str(sig) + "_covdiag_" + str(CovDiag)
histFileNm = FileNmBase + "HIST" + paraStr + ".png"
csvFileNm = FileNmBase + "DATA" + paraStr + ".csv"


#NumRuns =  1000 #of total runs
#NumSamps = 10000 #samples per run

NumRuns =  10 #of total runs
NumSamps = 1000 #samples per run



rho = .1
pSmp = 100

print("  ")
print("MTMpCN Run")
print("Total samples generated: " + str(NumRuns*NumSamps))

print("rho Value: " + str(rho))
print("p Value: " + str(pSmp))
print("  ")


badtryNm = 0




#Make mpCN Run Saving to .csv ensuring NAN errors do not stop the process

q0 = np.random.normal(0,1,NumParms)
for curRnInx in tqdm(range(0,NumRuns)):
    try:
        cursamps = MpCN_GPU(q0,NumParms,Cov,rho,Poty,pSmp,NumSamps +1)
        q0 = cursamps[NumSamps+1]
        writeCSV(csvFileNm,cursamps)
    except OverflowError:
        badtryNm = badtryNm +1

print("Number of failed runs: " + str(badtryNm))
print("Percentage of Failure: " + str(badtryNm/NumRuns))
        

  
MTMpCN Run
Total samples generated: 10000
rho Value: 0.1
p Value: 100
  


  0%|          | 0/10 [00:00<?, ?it/s]